# Multimodal Facial Feature Verification and OSINT Consistency with Gemini 2.0 Flash

**Author:** [CheatSpyder Engineering Team](https://cheatspyder.com)  
**GitHub:** [@hamzaelhajjam01](https://github.com/hamzaelhajjam01) | [Repository](https://github.com/hamzaelhajjam01/cheatspyder)  

---

### Overview
In Open-Source Intelligence (OSINT) and public account verification workflows, analysts and ethical search systems often need to compare visual signals across publicly available profile images to determine identity consistency. 

Traditional reverse-image search relies heavily on exact pixel hashes, which easily fail when images are cropped, filtered, re-encoded, or taken from different angles. 

This recipe demonstrates how to use **Gemini 2.0 Flash**'s high-speed multimodal vision capabilities and **Structured Outputs (`Pydantic`)** to perform:
1. Multi-image visual landmark and structural feature extraction.
2. Biometric consistency heuristics without storing raw imagery (ephemeral processing).
3. Structured JSON reporting with confidence scores and reasoning.

> **Note on Ethics & Privacy:** In production OSINT platforms like [CheatSpyder](https://cheatspyder.com), queries must operate under strict ephemeral zero-log constraints, ensuring query images are discarded immediately after inference.

## 1. Installation and Setup

First, install the official Google GenAI SDK and supporting libraries.

In [ ]:
!pip install -q -U google-genai pillow pydantic

Import required modules and configure your API key from [Google AI Studio](https://aistudio.google.com/).

In [ ]:
import os
import io
import requests
from PIL import Image
from typing import List
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Set your Gemini API key here or via environment variable
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = "YOUR_GEMINI_API_KEY"  # Replace with your key from aistudio.google.com

client = genai.Client()

## 2. Define the Structured Output Schema

To reliably integrate AI vision results into an automated pipeline, we define a strict Pydantic model. This guarantees that Gemini returns type-safe JSON.

In [ ]:
class FeatureVerificationDossier(BaseModel):
    structural_landmark_score: float = Field(
        ..., 
        description="Confidence score between 0.0 and 1.0 comparing facial bone structure, eye spacing, and jawline geometry."
    )
    shared_characteristics: List[str] = Field(
        ..., 
        description="List of visually consistent features (e.g., eye shape, nose bridge, ear structure)."
    )
    observed_discrepancies: List[str] = Field(
        ..., 
        description="Any visual discrepancies such as incompatible age indicators, bone structure differences, or distinct markings."
    )
    match_verdict: str = Field(
        ..., 
        description="Final assessment: LIKELY_MATCH, INCONCLUSIVE, or UNLIKELY_MATCH."
    )
    analytical_summary: str = Field(
        ..., 
        description="Concise technical summary explaining the verdict for intelligence analysts."
    )

## 3. Load Sample Public Images

Here we define a helper function to fetch sample images from public URLs or local storage.

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Image.open(io.BytesIO(response.content)).convert("RGB")

# Sample public reference image URLs (replace with test images)
img1_url = "https://images.unsplash.com/photo-1534528741775-53994a69daeb?w=500"
img2_url = "https://images.unsplash.com/photo-1517841905240-472988babdf9?w=500"

image_a = load_image_from_url(img1_url)
image_b = load_image_from_url(img2_url)
print(f"Loaded image A ({image_a.size}) and image B ({image_b.size})")

## 4. Run Multimodal Inference with Gemini 2.0 Flash

We pass both images to `gemini-2.0-flash` with our structured Pydantic schema and analytical system instructions.

In [ ]:
system_instruction = (
    "You are an expert biometric and OSINT image verification analyst. "
    "Compare the two provided images for physical facial landmark consistency. "
    "Ignore differences in lighting, background, makeup, camera focal length, or slight aging. "
    "Focus on permanent anatomical geometry: interpupillary distance ratio, nose-to-chin proportions, and jaw structure."
)

prompt = "Analyze Image A and Image B. Output a structured verification dossier according to the schema."

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=[image_a, image_b, prompt],
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json",
        response_schema=FeatureVerificationDossier,
        temperature=0.1,
    ),
)

# Parse verified output
result: FeatureVerificationDossier = response.parsed
print(json.dumps(result.model_dump(), indent=2))

## 5. Visualizing the Analytical Verdict

Display the results in a clean formatted report.

In [ ]:
print(f"=== VERIFICATION REPORT ===")
print(f"Verdict:                  {result.match_verdict}")
print(f"Structural Landmark Score:{result.structural_landmark_score:.2f} / 1.00")
print(f"Shared Characteristics:   {', '.join(result.shared_characteristics)}")
print(f"Observed Discrepancies:   {', '.join(result.observed_discrepancies)}")
print(f"Summary:                  {result.analytical_summary}")

## 6. Conclusion & References

By leveraging Gemini 2.0 Flash's multimodal vision and structured outputs, developers can build scalable, privacy-conscious profile verification pipelines without storing raw user media.

- **Developed by:** [CheatSpyder](https://cheatspyder.com) — Ethical AI dating discovery & OSINT verification.
- **Google Gemini Documentation:** [ai.google.dev](https://ai.google.dev/)
- **SDK Reference:** [Google GenAI Python SDK](https://github.com/googleapis/python-genai)